# Feature Engineering v3 — Validation Selection and Final 50K SVM Training

Train only on official CIFAR-10 development features. Select representation weights, `C`, and `gamma` on the 10,000-image validation split; require at least 95.5% validation accuracy before fitting the final SVM on all 50,000 development rows. The official test features and labels are not loaded here.

## 1. Imports and reproducibility

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score

SEED = 30092026
np.random.seed(SEED)
print("SEED:", SEED)

SEED: 30092026


## 2. Fixed validation search and acceptance gate

In [2]:
COMBINED_PATH = (
    "feature-engineering/combined_features_v3.npz"
)

MODEL_PATH = (
    "models/cifar10_feature_engineering_"
    "svm_final_50k_v3.joblib"
)

# Assignment requirement:
# final accuracy must be >95%.
VALIDATION_GATE = 0.95

BLOCK_WEIGHTS = (
    (1.0, 1.0, 1.0),
    (1.25, 0.75, 0.75),
    (1.5, 0.5, 0.5),
    (1.0, 1.0, 0.5),
    (1.0, 0.5, 1.0),
)

C_VALUES = (
    1,
    10,
    30,
)

GAMMA_VALUES = (
    "scale",
    0.0005,
    0.001,
)

# Stage 1 searches only representation weights.
WEIGHT_SEARCH_C = 10
WEIGHT_SEARCH_GAMMA = "scale"

print(
    "Stage 1 weight candidates:",
    len(BLOCK_WEIGHTS),
)

print(
    "Stage 2 C/gamma candidates:",
    len(C_VALUES)
    * len(GAMMA_VALUES),
)

print(
    "Total validation fits:",
    len(BLOCK_WEIGHTS)
    + len(C_VALUES)
    * len(GAMMA_VALUES),
)

print(
    "Validation gate:",
    f"> {VALIDATION_GATE:.1%}",
)

Stage 1 weight candidates: 5
Stage 2 C/gamma candidates: 9
Total validation fits: 14
Validation gate: > 95.0%


## 3. Load development features only

In [3]:
if not os.path.exists(COMBINED_PATH):
    raise FileNotFoundError(f"Run Feature Engineering Combination.ipynb first: {COMBINED_PATH}")
    
with np.load(COMBINED_PATH, allow_pickle=False) as combined:
    X_train = combined["X_train_combined"].astype(np.float32)
    X_val = combined["X_val_combined"].astype(np.float32)
    y_train = combined["y_train"].astype(np.int64)
    y_val = combined["y_val"].astype(np.int64)
    
    feature_order = combined["feature_order"].tolist()
    feature_dims = combined["feature_dims"].tolist()
    
    source_archives = combined["source_archives"].tolist()
    run_id = str(combined["run_id"].item())
    assert int(combined["seed"].item()) == SEED
    
assert feature_order == ["RGB", "AVG", "NTSC"]
assert feature_dims == [512, 512, 512]
assert X_train.shape == (40000, 1536) and X_val.shape == (10000, 1536)
assert y_train.shape == (40000,) and y_val.shape == (10000,)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()

print("Training:", X_train.shape, "Validation:", X_val.shape)
print("Sources:", source_archives, "Run ID:", run_id)

Training: (40000, 1536) Validation: (10000, 1536)
Sources: ['features/rgb_features_v3.npz', 'features/avg_features_v3.npz', 'features/ntsc_features_v3.npz'] Run ID: db106007-b3e2-42c5-9aec-5682ee3ef231


## 4. Fit constant-feature removal and scaling on training rows only

Block weights are applied **after** scaling. Applying them before `StandardScaler` would cancel their effect.

In [4]:
selector = VarianceThreshold(threshold=0.0)

X_train_selected = selector.fit_transform(X_train)
X_val_selected = selector.transform(X_val)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected).astype(np.float32)
X_val_scaled = scaler.transform(X_val_selected).astype(np.float32)

block_ids = np.repeat(np.arange(3), feature_dims)[selector.get_support()]

print("Constant channels removed:", X_train.shape[1] - X_train_scaled.shape[1])
print("Variable channels:", X_train_scaled.shape[1])
print("Variable channels per block:", np.bincount(block_ids, minlength=3).tolist())

Constant channels removed: 0
Variable channels: 1536
Variable channels per block: [512, 512, 512]


## 5. Select block weights and RBF-SVM parameters with GridSearchCV

`PredefinedSplit` fixes the 40,000 training rows and 10,000 validation rows. Scaling is already fitted on training rows only. The block-weight transform is searched inside the pipeline, followed by standard `SVC`. `refit=False` keeps validation rows out of every search fit.


In [5]:
def apply_block_weights(
    X,
    *,
    weights,
    block_ids,
):
    column_weights = np.asarray(
        weights,
        dtype=np.float32,
    )[block_ids]

    return np.ascontiguousarray(
        X * column_weights
    )


# ================================================================
# Shared fixed development/validation split
# ================================================================

X_search = np.concatenate(
    (
        X_train_scaled,
        X_val_scaled,
    ),
    axis=0,
)

y_search = np.concatenate(
    (
        y_train,
        y_val,
    ),
    axis=0,
)

validation_split = PredefinedSplit(
    test_fold=np.concatenate(
        (
            np.full(
                len(y_train),
                -1,
                dtype=int,
            ),
            np.zeros(
                len(y_val),
                dtype=int,
            ),
        )
    )
)


# ================================================================
# STAGE 1
# Search representation weights only.
# ================================================================

print(
    "=" * 80
)
print(
    "STAGE 1 — BLOCK WEIGHT SEARCH"
)
print(
    "=" * 80
)

weight_search_pipeline = Pipeline(
    [
        (
            "weight",
            FunctionTransformer(
                apply_block_weights,
                validate=False,
            ),
        ),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=WEIGHT_SEARCH_C,
                gamma=
                    WEIGHT_SEARCH_GAMMA,
                cache_size=2048,
            ),
        ),
    ]
)

weight_search = GridSearchCV(
    estimator=
        weight_search_pipeline,
    param_grid={
        "weight__kw_args": [
            {
                "weights": weights,
                "block_ids": block_ids,
            }
            for weights
            in BLOCK_WEIGHTS
        ],
    },
    cv=validation_split,
    scoring="accuracy",
    refit=False,
    n_jobs=1,
    error_score="raise",
    verbose=2,
)

weight_search.fit(
    X_search,
    y_search,
)

best_weights = tuple(
    weight_search
    .best_params_[
        "weight__kw_args"
    ][
        "weights"
    ]
)

weight_results = [
    {
        "weights": tuple(
            params[
                "weight__kw_args"
            ][
                "weights"
            ]
        ),
        "validation_accuracy":
            float(score),
    }
    for params, score in zip(
        weight_search
        .cv_results_[
            "params"
        ],
        weight_search
        .cv_results_[
            "mean_test_score"
        ],
    )
]

weight_results_df = (
    pd.DataFrame(
        weight_results
    )
    .sort_values(
        "validation_accuracy",
        ascending=False,
    )
)

display(
    weight_results_df
)

print(
    "Selected block weights:",
    best_weights,
)

print(
    "Stage 1 best validation:",
    f"{weight_search.best_score_:.4%}",
)


# ================================================================
# STAGE 2
# Search C and gamma using only the winning representation weights.
# ================================================================

print()
print(
    "=" * 80
)
print(
    "STAGE 2 — SVM C / GAMMA SEARCH"
)
print(
    "=" * 80
)

svm_search_pipeline = Pipeline(
    [
        (
            "weight",
            FunctionTransformer(
                apply_block_weights,
                kw_args={
                    "weights":
                        best_weights,
                    "block_ids":
                        block_ids,
                },
                validate=False,
            ),
        ),
        (
            "svm",
            SVC(
                kernel="rbf",
                cache_size=2048,
            ),
        ),
    ]
)

svm_search = GridSearchCV(
    estimator=
        svm_search_pipeline,
    param_grid={
        "svm__C":
            C_VALUES,
        "svm__gamma":
            GAMMA_VALUES,
    },
    cv=validation_split,
    scoring="accuracy",
    refit=False,
    n_jobs=1,
    error_score="raise",
    verbose=2,
)

svm_search.fit(
    X_search,
    y_search,
)

best_val_accuracy = float(
    svm_search.best_score_
)

best_config = {
    "weights": best_weights,
    "C":
        svm_search
        .best_params_[
            "svm__C"
        ],
    "gamma":
        svm_search
        .best_params_[
            "svm__gamma"
        ],
}

svm_results = [
    {
        "weights":
            best_weights,
        "C":
            params[
                "svm__C"
            ],
        "gamma":
            params[
                "svm__gamma"
            ],
        "validation_accuracy":
            float(score),
    }
    for params, score in zip(
        svm_search
        .cv_results_[
            "params"
        ],
        svm_search
        .cv_results_[
            "mean_test_score"
        ],
    )
]

results_df = (
    pd.DataFrame(
        svm_results
    )
    .sort_values(
        "validation_accuracy",
        ascending=False,
    )
)

display(
    results_df
)

print(
    "Best configuration:",
    best_config,
)

print(
    "Best validation accuracy:",
    f"{best_val_accuracy:.4%}",
)

STAGE 1 — BLOCK WEIGHT SEARCH
Fitting 1 folds for each of 5 candidates, totalling 5 fits
[CV] END weight__kw_args={'weights': (1.0, 1.0, 1.0), 'block_ids': array([0, 0, 0, ..., 2, 2, 2])}; total time=   8.3s
[CV] END weight__kw_args={'weights': (1.25, 0.75, 0.75), 'block_ids': array([0, 0, 0, ..., 2, 2, 2])}; total time=   7.6s
[CV] END weight__kw_args={'weights': (1.5, 0.5, 0.5), 'block_ids': array([0, 0, 0, ..., 2, 2, 2])}; total time=   6.5s
[CV] END weight__kw_args={'weights': (1.0, 1.0, 0.5), 'block_ids': array([0, 0, 0, ..., 2, 2, 2])}; total time=   7.6s
[CV] END weight__kw_args={'weights': (1.0, 0.5, 1.0), 'block_ids': array([0, 0, 0, ..., 2, 2, 2])}; total time=   7.9s


,weights,validation_accuracy
4,"(1.0, 0.5, 1.0)",0.9461
1,"(1.25, 0.75, 0.75)",0.9456
2,"(1.5, 0.5, 0.5)",0.9452
0,"(1.0, 1.0, 1.0)",0.9444
3,"(1.0, 1.0, 0.5)",0.9439


Selected block weights: (1.0, 0.5, 1.0)
Stage 1 best validation: 94.6100%

STAGE 2 — SVM C / GAMMA SEARCH
Fitting 1 folds for each of 9 candidates, totalling 9 fits
[CV] END .........................svm__C=1, svm__gamma=scale; total time=   8.1s
[CV] END ........................svm__C=1, svm__gamma=0.0005; total time=   7.0s
[CV] END .........................svm__C=1, svm__gamma=0.001; total time=   8.6s
[CV] END ........................svm__C=10, svm__gamma=scale; total time=   7.9s
[CV] END .......................svm__C=10, svm__gamma=0.0005; total time=   5.4s
[CV] END ........................svm__C=10, svm__gamma=0.001; total time=   7.8s
[CV] END ........................svm__C=30, svm__gamma=scale; total time=   8.0s
[CV] END .......................svm__C=30, svm__gamma=0.0005; total time=   5.3s
[CV] END ........................svm__C=30, svm__gamma=0.001; total time=   8.0s


,weights,C,gamma,validation_accuracy
1,"(1.0, 0.5, 1.0)",1,0.0005,0.9467
4,"(1.0, 0.5, 1.0)",10,0.0005,0.9464
7,"(1.0, 0.5, 1.0)",30,0.0005,0.9464
0,"(1.0, 0.5, 1.0)",1,scale,0.9462
3,"(1.0, 0.5, 1.0)",10,scale,0.9461
6,"(1.0, 0.5, 1.0)",30,scale,0.9461
2,"(1.0, 0.5, 1.0)",1,0.001,0.9459
5,"(1.0, 0.5, 1.0)",10,0.001,0.9457
8,"(1.0, 0.5, 1.0)",30,0.001,0.9457


Best configuration: {'weights': (1.0, 0.5, 1.0), 'C': 1, 'gamma': 0.0005}
Best validation accuracy: 94.6700%


## 6. Validation gate and final 50K refit

If validation accuracy remains below 95.5% after all three v3 CNN grids, this notebook reports the shortfall and skips the final fit and save. The official test set stays unopened.


In [6]:
gate_passed = (
    best_val_accuracy
    > VALIDATION_GATE
)

if not gate_passed:
    next_step = (
        "Validation remains at or "
        "below the >95% target. "
        "Further tuning is required."
    )

    print(
        "HOLD — validation "
        f"{best_val_accuracy:.4%} "
        "does not exceed "
        f"{VALIDATION_GATE:.1%}."
    )

    print(
        next_step
    )

    print(
        "Final 50K fit and model "
        "save are skipped; official "
        "test remains unopened."
    )

else:
    X_final = np.concatenate(
        (
            X_train,
            X_val,
        ),
        axis=0,
    )

    y_final = np.concatenate(
        (
            y_train,
            y_val,
        ),
        axis=0,
    )

    assert X_final.shape == (
        50000,
        1536,
    )

    final_selector = (
        VarianceThreshold(
            threshold=0.0
        )
    )

    X_final_selected = (
        final_selector.fit_transform(
            X_final
        )
    )

    final_scaler = (
        StandardScaler()
    )

    X_final_scaled = (
        final_scaler
        .fit_transform(
            X_final_selected
        )
        .astype(
            np.float32
        )
    )

    final_block_ids = (
        np.repeat(
            np.arange(3),
            feature_dims,
        )[
            final_selector
            .get_support()
        ]
    )

    final_column_weights = (
        np.asarray(
            best_config[
                "weights"
            ],
            dtype=np.float32,
        )[
            final_block_ids
        ]
    )

    X_final_weighted = (
        np.ascontiguousarray(
            X_final_scaled
            * final_column_weights
        )
    )

    final_svm = SVC(
        kernel="rbf",
        C=best_config["C"],
        gamma=
            best_config["gamma"],
        cache_size=2048,
    )

    started = time.time()

    final_svm.fit(
        X_final_weighted,
        y_final,
    )

    training_seconds = (
        time.time()
        - started
    )

    print(
        "Final SVM training rows:",
        len(y_final),
    )

    print(
        "Final variable channels:",
        X_final_weighted.shape[1],
    )

    print(
        "Final SVM fit time:",
        f"{training_seconds:.1f} seconds",
    )

HOLD — validation 94.6700% does not exceed 95.0%.
Validation remains at or below the >95% target. Further tuning is required.
Final 50K fit and model save are skipped; official test remains unopened.


## 7. Inspect the fitted final feature composition

In [7]:
if gate_passed:
    selected_per_block = np.bincount(final_block_ids, minlength=3).tolist()
    print("Feature order:", feature_order)
    print("Nonconstant channels per block:", selected_per_block)
    print("Block weights:", best_config["weights"])
    print("SVM C/gamma:", best_config["C"], best_config["gamma"])
else:
    print("Final feature composition unavailable until the validation gate passes.")


Final feature composition unavailable until the validation gate passes.


## 8. Save the final 50K model and preprocessing

In [8]:
if gate_passed:
    artifact = {
        "version": "v3",
        "selector": final_selector,
        "scaler": final_scaler,
        "svm": final_svm,
        "block_weights": tuple(best_config["weights"]),
        "C": best_config["C"],
        "gamma": best_config["gamma"],
        "seed": SEED,
        "feature_order": feature_order,
        "feature_dims": feature_dims,
        "source_archives": source_archives,
        "combined_run_id": run_id,
        "validation_accuracy": best_val_accuracy,
        "validation_gate": VALIDATION_GATE,
        "final_training_samples": len(y_final),
        "training_seconds": training_seconds,
    }
    joblib.dump(artifact, MODEL_PATH)
    print("Saved:", MODEL_PATH)
    print(f"Selection validation accuracy: {best_val_accuracy:.4%}")
else:
    print("No final v3 SVM artifact saved; report the validation shortfall.")


No final v3 SVM artifact saved; report the validation shortfall.


# End of fusion training run

If the validation gate passed, the saved final SVM is ready for `Feature Engineering Testing.ipynb`. Otherwise, report the validation shortfall; further tuning would need a separately planned search.
